# MovieLens Step-by-Step Verification Notebook (v5 기준)

목표: `main_experiment_grid_search_0116_v5.ipynb` 실험이 의도대로 수행되었는지
**실제 MovieLens 데이터**를 단계별로 눈으로 확인한다.

검증 Step
1. v5 설정값 확인 (fold, K, TopN, λ)
2. fold별 split 파일 구조 확인
3. train_inner/validation/test leakage 점검
4. 결과 파일의 조건 일치 확인
5. 실제 α 선택 로직(regularized score) 재검산
6. baseline(α=1) vs optimized 성능 비교
7. 최종 체크리스트(PASS/FAIL)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# ===== v5 기준 설정 =====
FOLDS_TO_RUN = [8, 9, 10]
K_RANGE_EXPECTED = list(range(10, 101, 10))
TOPN_RANGE_EXPECTED = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
REGULARIZATION_LAMBDA = 0.002
METHOD_COUNT_EXPECTED = 17

root = Path.cwd().resolve()
project_root = root.parent if root.name == 'notebooks' else root
data_root = project_root / 'data' / 'movielenz_data'
results_root = project_root / 'results'

print('[v5 설정 확인]')
print('Project root:', project_root)
print('Folds:', FOLDS_TO_RUN)
print('K range:', K_RANGE_EXPECTED)
print('TopN range:', TOPN_RANGE_EXPECTED)
print('Lambda:', REGULARIZATION_LAMBDA)
print('Expected methods:', METHOD_COUNT_EXPECTED)

[v5 설정 확인]
Project root: D:\Dropbox\python_workspace(백업)\imputation_project
Folds: [8, 9, 10]
K range: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
TopN range: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
Lambda: 0.002
Expected methods: 17


In [2]:
# Step 1) split 파일 로드 및 기본 통계
required = ['train_inner.csv', 'validation.csv', 'train.csv', 'test.csv']
splits = {}
rows = []

for fold in FOLDS_TO_RUN:
    fid = f'fold_{fold:02d}'
    fdir = data_root / fid
    splits[fold] = {}

    for fname in required:
        path = fdir / fname
        exists = path.exists()
        if exists:
            df = pd.read_csv(path, index_col=0)
            splits[fold][fname] = df
            observed = int(df.notna().sum().sum())
            rows.append({
                'fold': fold,
                'file': fname,
                'shape': f'{df.shape[0]}x{df.shape[1]}',
                'observed_cells': observed,
                'sparsity_%': round(100 * (1 - observed / df.size), 2)
            })
        else:
            rows.append({'fold': fold, 'file': fname, 'shape': 'MISSING', 'observed_cells': None, 'sparsity_%': None})

split_stats = pd.DataFrame(rows).sort_values(['fold', 'file']).reset_index(drop=True)
display(split_stats)

print('\n[샘플 확인: fold 08 train_inner 상위 5행 x 6열]')
display(splits[8]['train_inner.csv'].iloc[:5, :6])

,fold,file,shape,observed_cells,sparsity_%
0,8,test.csv,1682x943,9738,99.39
1,8,train.csv,1682x943,90262,94.31
2,8,train_inner.csv,1682x943,72189,95.45
3,8,validation.csv,1682x943,18073,98.86
4,9,test.csv,1682x943,9652,99.39
5,9,train.csv,1682x943,90348,94.30
6,9,train_inner.csv,1682x943,72259,95.44
7,9,validation.csv,1682x943,18089,98.86
8,10,test.csv,1682x943,9587,99.40
9,10,train.csv,1682x943,90413,94.30



[샘플 확인: fold 08 train_inner 상위 5행 x 6열]


,1,2,3,4,5,6
1,5.0,4.0,NaN,NaN,4.0,4.0
2,3.0,NaN,NaN,NaN,3.0,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN
4,3.0,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Step 2) leakage 점검
def observed_coords(df):
    coords = np.argwhere((~df.isna()).values)
    return set((int(r), int(c)) for r, c in coords)

checks = []
for fold in FOLDS_TO_RUN:
    d = splits[fold]
    c_inner = observed_coords(d['train_inner.csv'])
    c_val = observed_coords(d['validation.csv'])
    c_train = observed_coords(d['train.csv'])
    c_test = observed_coords(d['test.csv'])

    ov_inner_val = len(c_inner & c_val)
    ov_inner_test = len(c_inner & c_test)
    ov_val_test = len(c_val & c_test)

    union_inner_val = c_inner | c_val
    train_extra = len(c_train - union_inner_val)
    train_missing = len(union_inner_val - c_train)

    checks.append({
        'fold': fold,
        'overlap(train_inner,validation)': ov_inner_val,
        'overlap(train_inner,test)': ov_inner_test,
        'overlap(validation,test)': ov_val_test,
        'train_vs(inner+val)_extra': train_extra,
        'train_vs(inner+val)_missing': train_missing
    })

leakage_df = pd.DataFrame(checks)
display(leakage_df)

leakage_pass = (
    (leakage_df[['overlap(train_inner,validation)','overlap(train_inner,test)','overlap(validation,test)']] == 0).all().all()
    and (leakage_df[['train_vs(inner+val)_extra','train_vs(inner+val)_missing']] == 0).all().all()
)
print('✅ Leakage check PASS' if leakage_pass else '❌ Leakage check FAIL')

,fold,"overlap(train_inner,validation)","overlap(train_inner,test)","overlap(validation,test)",train_vs(inner+val)_extra,train_vs(inner+val)_missing
0,8,0,0,0,0,0
1,9,0,0,0,0,0
2,10,0,0,0,0,0


✅ Leakage check PASS


In [4]:
# Step 3) v5 결과 파일 로딩 및 조건 일치 점검
grid_files = {
    8: results_root / 'fold_08' / 'grid_search_results_20260128_222923.csv',
    9: results_root / 'fold_09' / 'grid_search_results_20260129_023436.csv',
    10: results_root / 'fold_10' / 'grid_search_results_20260129_063658.csv',
}
alpha_hist_files = {
    8: results_root / 'fold_08' / 'alpha_optimization_history_20260128_222923.csv',
    9: results_root / 'fold_09' / 'alpha_optimization_history_20260129_023436.csv',
    10: results_root / 'fold_10' / 'alpha_optimization_history_20260129_063658.csv',
}

grid = {f: pd.read_csv(p) for f, p in grid_files.items()}
hist = {f: pd.read_csv(p) for f, p in alpha_hist_files.items()}

meta_rows = []
for fold in FOLDS_TO_RUN:
    df = grid[fold]
    meta_rows.append({
        'fold': fold,
        'rows': len(df),
        'methods': df['method'].nunique(),
        'K_ok': sorted(df['K'].unique().tolist()) == K_RANGE_EXPECTED,
        'TopN_ok': sorted(df['TopN'].unique().tolist()) == TOPN_RANGE_EXPECTED,
        'types': ','.join(sorted(df['type'].unique().tolist())),
        'alpha_hist_rows': len(hist[fold])
    })

meta_df = pd.DataFrame(meta_rows)
display(meta_df)

conditions_pass = (
    (meta_df['methods'] == METHOD_COUNT_EXPECTED).all() and
    meta_df['K_ok'].all() and
    meta_df['TopN_ok'].all() and
    meta_df['types'].str.contains('baseline').all() and
    meta_df['types'].str.contains('optimized').all()
)
print('✅ v5 조건 일치 PASS' if conditions_pass else '❌ v5 조건 일치 FAIL')

,fold,rows,methods,K_ok,TopN_ok,types,alpha_hist_rows
0,8,3400,17,True,True,"baseline,optimized",55320
1,9,3400,17,True,True,"baseline,optimized",55620
2,10,3400,17,True,True,"baseline,optimized",54900


✅ v5 조건 일치 PASS


In [5]:
# Step 4) α 선택 로직 재검산 (fold 08, method=pcc, K=20, TopN=10 예시)
fold = 8
method = 'pcc'
K = 20
TopN = 10

h = hist[fold]
g = grid[fold]

h_sub = h[(h['method'] == method) & (h['K'] == K) & (h['TopN'] == TopN)].copy()
h_sub = h_sub.sort_values('regularized_score')

g_opt = g[(g['method'] == method) & (g['K'] == K) & (g['TopN'] == TopN) & (g['type'] == 'optimized')].iloc[0]
g_base = g[(g['method'] == method) & (g['K'] == K) & (g['TopN'] == TopN) & (g['type'] == 'baseline')].iloc[0]

best_hist = h_sub.iloc[0]

print('[검증 대상] fold=8, method=pcc, K=20, TopN=10')
print('best alpha from history:', best_hist['alpha'])
print('optimized alpha in grid :', g_opt['alpha'])
print('same alpha? ->', np.isclose(best_hist['alpha'], g_opt['alpha']))

print('\nTop 10 candidates by regularized_score:')
display(h_sub[['phase','alpha','mse','regularization_penalty','regularized_score','rmse','precision','recall']].head(10))

print('\nBaseline vs Optimized (same fold/method/K/TopN):')
display(pd.DataFrame([{
    'baseline_alpha': g_base['alpha'],
    'baseline_test_RMSE': g_base['test_RMSE'],
    'optimized_alpha': g_opt['alpha'],
    'optimized_test_RMSE': g_opt['test_RMSE'],
    'delta_opt_minus_base': g_opt['test_RMSE'] - g_base['test_RMSE']
}]))

[검증 대상] fold=8, method=pcc, K=20, TopN=10
best alpha from history: 3.0
optimized alpha in grid : 3.0
same alpha? -> True

Top 10 candidates by regularized_score:


,phase,alpha,mse,regularization_penalty,regularized_score,rmse,precision,recall
481,coarse,3.00,0.687111,0.0040,0.691111,0.828922,0.715140,0.803280
731,fine,3.00,0.687111,0.0040,0.691111,0.828922,0.715140,0.803280
721,fine,2.95,0.687316,0.0039,0.691216,0.829045,0.715034,0.803198
711,fine,2.90,0.687546,0.0038,0.691346,0.829184,0.714822,0.803047
491,coarse,3.50,0.686372,0.0050,0.691372,0.828475,0.715458,0.803359
701,fine,2.85,0.687800,0.0037,0.691500,0.829337,0.714716,0.803022
691,fine,2.80,0.688079,0.0036,0.691679,0.829505,0.714928,0.803099
681,fine,2.75,0.688382,0.0035,0.691882,0.829688,0.714716,0.802981
671,fine,2.70,0.688711,0.0034,0.692111,0.829886,0.714928,0.803102
661,fine,2.65,0.689065,0.0033,0.692365,0.830099,0.715034,0.803132



Baseline vs Optimized (same fold/method/K/TopN):


,baseline_alpha,baseline_test_RMSE,optimized_alpha,optimized_test_RMSE,delta_opt_minus_base
0,1.0,1.023513,3.0,1.04043,0.016917


In [6]:
# Step 5) fold별 평균 성능 요약 (v5 범위: 8,9,10)
rows = []
for fold in FOLDS_TO_RUN:
    df = grid[fold]
    base_rmse = df[df['type'] == 'baseline']['test_RMSE'].mean()
    opt_rmse = df[df['type'] == 'optimized']['test_RMSE'].mean()
    rows.append({
        'fold': fold,
        'baseline_rmse_mean': base_rmse,
        'optimized_rmse_mean': opt_rmse,
        'delta_%': 100*(opt_rmse-base_rmse)/base_rmse
    })

fold_summary = pd.DataFrame(rows).sort_values('fold')
display(fold_summary)

overall_base = fold_summary['baseline_rmse_mean'].mean()
overall_opt = fold_summary['optimized_rmse_mean'].mean()
overall_delta = 100*(overall_opt-overall_base)/overall_base

print(f'Overall (fold 8-10) baseline mean RMSE : {overall_base:.6f}')
print(f'Overall (fold 8-10) optimized mean RMSE: {overall_opt:.6f}')
print(f'Overall delta (opt-base) %          : {overall_delta:.3f}%')

,fold,baseline_rmse_mean,optimized_rmse_mean,delta_%
0,8,1.027395,1.044802,1.694216
1,9,1.005957,1.026370,2.029250
2,10,1.012023,1.030954,1.870600


Overall (fold 8-10) baseline mean RMSE : 1.015125
Overall (fold 8-10) optimized mean RMSE: 1.034042
Overall delta (opt-base) %          : 1.864%


In [7]:
# Step 6) 최종 PASS/FAIL 체크리스트
checklist = pd.DataFrame([
    {'check': 'split files available', 'pass': (split_stats['shape'] != 'MISSING').all()},
    {'check': 'no leakage across split', 'pass': leakage_pass},
    {'check': 'v5 condition match', 'pass': conditions_pass},
    {'check': 'alpha selection reproducible (sample)', 'pass': bool(np.isclose(best_hist['alpha'], g_opt['alpha']))},
])
display(checklist)

if checklist['pass'].all():
    print('🎉 FINAL VERDICT: PASS')
    print('실데이터(MovieLens) 기준으로 v5 실험이 의도대로 동작했음을 단계별로 확인했습니다.')
else:
    print('❌ FINAL VERDICT: CHECK REQUIRED')
    print('실패 항목을 확인한 뒤 교수님 보고 자료에 반영하세요.')

,check,pass
0,split files available,True
1,no leakage across split,True
2,v5 condition match,True
3,alpha selection reproducible (sample),True


🎉 FINAL VERDICT: PASS
실데이터(MovieLens) 기준으로 v5 실험이 의도대로 동작했음을 단계별로 확인했습니다.
